# Chapter 1. 멀티모달 이해 더 알아보기

## 1.1 비디오로 확장된 멀티모달 모델

### 1.1.2. 비디오 LLM의 핵심과제

In [2]:
!pip install -q num2words

In [3]:
from huggingface_hub import hf_hub_download

video_path = hf_hub_download(
   repo_id="woojun-jung/multimodal-guide-using-huggingface",
   filename="Part-5/dog_walking.mp4",
   repo_type="dataset",
)

In [4]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

model_id = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
   model_id,
   device_map="auto",
   dtype="auto"
)

conversation = [
   {
       "role": "user",
       "content": [
           {"type": "video", "path": video_path},
           {"type": "text", "text": "Describe this video in detail."},
       ],
   },
]

inputs = processor.apply_chat_template(
   conversation,
   add_generation_prompt=True,
   tokenize=True,
   return_dict=True,
   return_tensors="pt",
).to(device=model.device, dtype=model.dtype)

generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=256)
print(processor.batch_decode(generated_ids, skip_special_tokens=True)[0])

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

You have used fast image processor with LANCZOS resample which not yet supported for torch.Tensor. BICUBIC resample will be used as an alternative. Please fall back to image processor if you want full consistency with the original model.


User: You are provided the following series of twenty-two frames from a 0:00:22 [H:MM:SS] video.

Frame from 00:00:
Frame from 00:01:
Frame from 00:02:
Frame from 00:03:
Frame from 00:04:
Frame from 00:05:
Frame from 00:06:
Frame from 00:07:
Frame from 00:08:
Frame from 00:09:
Frame from 00:10:
Frame from 00:11:
Frame from 00:12:
Frame from 00:13:
Frame from 00:14:
Frame from 00:15:
Frame from 00:16:
Frame from 00:17:
Frame from 00:18:
Frame from 00:19:
Frame from 00:20:
Frame from 00:21:

Describe this video in detail.
Assistant: The video features a black dog with a harness walking on a stone path in a garden, with a leash attached to its collar. The dog is seen walking along the path, which is bordered by a hedge and a tree, and is accompanied by a person whose hand is visible in the frame. The dog appears to be on a leash, as it is being guided by the person. The garden is well-maintained, with neatly trimmed bushes and a variety of plants. The path is made of flat, rectangular sto

## 1.2. 모든 모달리티를 하나로: Omni 모델

### 1.2.1. Omni 모델

In [5]:
!pip install qwen-omni-utils[decord] -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 132.1 MB/s eta 0:00:00


In [7]:
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor, AutoConfig
from qwen_omni_utils import process_mm_info

model_id = "Qwen/Qwen2.5-Omni-3B"

# config 먼저 로드
config = AutoConfig.from_pretrained(model_id)

# TalkerConfig에 pad_token_id가 없으면 text_config에서 가져와서 세팅
if not hasattr(config.talker_config, 'pad_token_id'):
    config.talker_config.pad_token_id = config.thinker_config.text_config.pad_token_id

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
   model_id,
   config=config,
   dtype="auto",
   device_map="auto"
)
processor = Qwen2_5OmniProcessor.from_pretrained(model_id)

Loading weights:   0%|          | 0/2543 [00:00<?, ?it/s]

Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
messages = [
   {
       "role": "user",
       "content": [
           {"type": "audio", "audio": "https://huggingface.co/datasets/huggingface-KREW/multimodal-book/resolve/main/Part-5/train_sound.mp3"},
           {"type": "text", "text": "What do you hear in this audio?"},
       ],
   },
]

# 1) 텍스트 템플릿 생성
text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

In [9]:
# 2) 멀티모달 데이터 로드 및 전처리
audios, images, videos = process_mm_info(messages, use_audio_in_video=False)

# 3) 모든 입력을 합쳐서 모델 입력 생성
inputs = processor(
   text=text, audio=audios, images=images, videos=videos,
   return_tensors="pt", padding=True
)
inputs = inputs.to(device=model.device, dtype=model.dtype)

text_ids = model.generate(**inputs, return_audio=False, max_new_tokens=100)
input_len = inputs["input_ids"].shape[-1]
output_ids = text_ids[:, input_len:]

print(processor.decode(output_ids, skip_special_tokens=True)[0])

/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:172: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In this audio, you can hear a train horn blowing, followed by a bell ringing, and then the sound of a train passing by.
